#### Imports

In [ ]:
import importlib
import pandas as pd
import numpy as np
import random

from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.gaussian_process import GaussianProcessRegressor

import globals
import utils

np.set_printoptions(precision=4, suppress=True, linewidth=np.inf, threshold=np.inf)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)

seed = 42
random.seed(seed)
np.random.seed(seed)

In [2]:
importlib.reload(globals)
importlib.reload(utils)

<module 'utils' from 'c:\\Users\\Matteo\\Desktop\\Scuola\\MastersThesis\\Masters_Thesis\\utils.py'>

In [3]:
param_names, function_names = utils.inspect_metadata(globals.CURRENT_TRAIN_FILE)

Keys in train_file: ['I0', 'LUTdata', 'LUTheader', 'dynamic', 'static', 'wvl']

Attributes in LUTheader (inputs):
  varnames: O3STR,H2OSTR,VIS,G,ASTMX,SSA,PARM2,OBSZEN,PARM1

Attributes in train_file (outputs):
  RTMname: MODTRAN6
  inputmode: Latin hypercube
  lut_package_date: 10-Apr-2025
  opmode: Transfer Functions
  outnames: Lp0,Edir0,Edif0,S,tdir,tdif
  sensor: Empty(dtype=dtype('O'))

LUTheader shape: (500, 9)
LUTdata shape: (500, 25230)
wvl shape: (1, 4205)


In [4]:
X, Y, wavelengths = utils.load_train_h5(globals.CURRENT_TRAIN_FILE)
wavelengths = wavelengths.squeeze()
Y_resh = Y.reshape(-1, Y.shape[1] // len(wavelengths), len(wavelengths))

X_tr, X_val, X_test, Y_tr, Y_val, Y_test = utils.train_val_test_split(X, Y_resh, wavelengths, verbose=True)

X shape: (500, 9)
Y shape: (500, 6, 4205)
wavelengths shape: (4205,)

X_tr shape: (400, 9)
X_val shape: (50, 9)
X_test shape: (50, 9)

Y_tr shape: (400, 6, 4205)
Y_val shape: (50, 6, 4205)
Y_test shape: (50, 6, 4205)


#### Gaussian Processes on each Wavelength (no PCA)

GPs deal with the high number of bands with PCA, reducing them to just 10 Principal Components. However, PCA lacks interpretability. The following section investigates a method to avoid using dimensionality reduction while still keeping computational times feasible.

In [5]:
# ----- configuration -----
scale_type_piecewise = "minmax"

degree_piecewise = 10
window_size_nm = 7

print("Taylor degree:", degree_piecewise)
print("Window size:", window_size_nm)


def create_spectral_windows(wavelengths, window_size_nm):
    windows = []
    start = wavelengths.min()

    while start < wavelengths.max():
        idx = np.where((wavelengths >= start) & (wavelengths < start + window_size_nm))[0]

        if len(idx) > 0:
            windows.append(idx)

        start += window_size_nm

    return windows

spectral_windows = create_spectral_windows(wavelengths, window_size_nm)
print("Number of spectral windows:", len(spectral_windows))


def spectra_to_piecewise(Y, windows, degree, wavelengths):
    """
    Convert spectra into local polynomial coefficients.

    output:
    (samples, functions, windows*(degree+1))
    """

    coeffs_all = []

    for idx in windows:
        wl = wavelengths[idx]
        x = (wl - wl.mean()) / wl.std()

        A = np.polynomial.chebyshev.chebvander(x, degree)

        coeffs_window = []

        for f in range(Y.shape[1]):
            y = Y[:, f, idx]

            c = np.linalg.lstsq(A, y.T, rcond=None)[0].T

            coeffs_window.append(c)

        coeffs_window = np.stack(coeffs_window, axis=1)
        coeffs_all.append(coeffs_window)

    coeffs_all = np.stack(coeffs_all, axis=2)

    # flatten windows and coefficients
    coeffs_all = coeffs_all.reshape(Y.shape[0], Y.shape[1], -1)

    return coeffs_all


def piecewise_to_spectrum(coefficients, windows, degree, wavelengths):
    """
    Reconstruct spectra from piecewise polynomial coefficients.
    """

    n_samples = coefficients.shape[0]
    n_functions = coefficients.shape[1]

    Y = np.zeros((n_samples, n_functions, len(wavelengths)))

    coeff_idx = 0

    for idx in windows:

        wl = wavelengths[idx]
        x = (wl - wl.mean()) / wl.std()

        A = np.polynomial.chebyshev.chebvander(x, degree)

        for f in range(n_functions):

            coeff = coefficients[:, f, coeff_idx:coeff_idx+degree+1]

            Y[:, f, idx] = coeff @ A.T

        coeff_idx += degree + 1

    return Y

Taylor degree: 10
Window size: 7
Number of spectral windows: 301


In [6]:
min_mre = 9999
min_mae = 9999
min_degree = 9999
min_window_sz = 9999

for degree in range(2, 11):
    for window_sz in [7, 10, 20, 50, 100, 200, 500]:
        windows = create_spectral_windows(wavelengths, window_sz)
        Y_test_coeff = spectra_to_piecewise(Y_val, windows, degree, wavelengths)
        Y_test_recon = np.zeros_like(Y_val)

        Y_test_recon = piecewise_to_spectrum(Y_test_coeff, windows, degree, wavelengths)

        mre_recon = utils.mre_score(Y_val, Y_test_recon, wavelengths, axis=None)
        mae_recon = utils.mae_score(Y_val, Y_test_recon, wavelengths, axis=None)

        print(f"{degree} | {window_sz} | MRE: {mre_recon:.4f} | MAE: {mae_recon:.4f}")

        if mre_recon < min_mre:
            min_mre = mre_recon
            min_mae = mae_recon
            min_degree = degree
            min_window_sz = window_sz

print(f"\nBest result:\n\tMRE: {min_mre:.4f} | MAE: {min_mae:.4f} | Degree: {min_degree} | Window size: {min_window_sz}")

2 | 7 | MRE: 0.0556 | MAE: 15.9087
2 | 10 | MRE: 0.0576 | MAE: 16.1364
2 | 20 | MRE: 0.0621 | MAE: 16.6033
2 | 50 | MRE: 0.0729 | MAE: 17.6362
2 | 100 | MRE: 0.0820 | MAE: 18.4324
2 | 200 | MRE: 0.0970 | MAE: 19.5186
2 | 500 | MRE: 0.1280 | MAE: 21.8858
3 | 7 | MRE: 0.0536 | MAE: 15.5773
3 | 10 | MRE: 0.0557 | MAE: 15.8070
3 | 20 | MRE: 0.0605 | MAE: 16.4561
3 | 50 | MRE: 0.0705 | MAE: 17.4875
3 | 100 | MRE: 0.0770 | MAE: 17.9583
3 | 200 | MRE: 0.0874 | MAE: 18.8875
3 | 500 | MRE: 0.1048 | MAE: 19.8942
4 | 7 | MRE: 0.0517 | MAE: 15.3497
4 | 10 | MRE: 0.0541 | MAE: 15.6022
4 | 20 | MRE: 0.0590 | MAE: 16.3092
4 | 50 | MRE: 0.0680 | MAE: 17.1690
4 | 100 | MRE: 0.0750 | MAE: 17.8323
4 | 200 | MRE: 0.0854 | MAE: 18.7471
4 | 500 | MRE: 0.1035 | MAE: 19.6351
5 | 7 | MRE: 0.0503 | MAE: 15.1339
5 | 10 | MRE: 0.0527 | MAE: 15.4180
5 | 20 | MRE: 0.0581 | MAE: 16.1763
5 | 50 | MRE: 0.0672 | MAE: 17.0414
5 | 100 | MRE: 0.0732 | MAE: 17.7486
5 | 200 | MRE: 0.0795 | MAE: 18.1976
5 | 500 | MRE: 0.1014

Reconstruction MRE is already too high, with a degree and window size that render the GP fitting computationally unfeasible as well.

In [ ]:
# Y_tr_piecewise = spectra_to_piecewise(Y_tr, spectral_windows, degree_piecewise, wavelengths)
# Y_val_piecewise = spectra_to_piecewise(Y_val, spectral_windows, degree_piecewise, wavelengths)
# print("Piecewise coefficient shape training:", Y_tr_piecewise.shape)
# print("Piecewise coefficient shape validation:", Y_val_piecewise.shape)

# x_scaler_piecewise, X_tr_scaled_piecewise, X_val_scaled_piecewise = utils.scale_input_data(X_tr, X_val, scale_type=scale_type_piecewise)
# y_scalers_piecewise, Y_tr_piecewise_scaled, Y_val_piecewise_scaled = utils.scale_output_data(Y_tr_piecewise, Y_val_piecewise, scale_type=scale_type_piecewise)

# piecewise_kernel = (
#     C(1.0, (1e-3,1e3))
#     *
#     Matern(length_scale=np.ones(globals.N_INPUTS), nu=2.5, length_scale_bounds=(1e-3,1e3))
#     +
#     WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5,1e1))
# )

Piecewise coefficient shape training: (400, 6, 3311)
Piecewise coefficient shape validation: (50, 6, 3311)
---------- Scaling input data using minmax scaling... ----------
---------- Input data scaling completed. ----------

---------- Scaling output data using minmax scaling... ----------
---------- Output data scaling completed. ----------



In [ ]:
# piecewise_gp, piecewise_time = fit_gp(X_tr_scaled_piecewise, Y_tr_piecewise_scaled, piecewise_kernel)

In [ ]:
# def validate_piecewise_taylor(gpr_list, X_scaled, y_scalers):
#     Y_pred = np.zeros_like(Y_val)

#     for f_idx in range(globals.N_FUNCTIONS):

#         coeff_scaled = gpr_list[f_idx].predict(X_scaled)
#         coeff = y_scalers[f_idx].inverse_transform(coeff_scaled)

#         spectrum = piecewise_to_spectrum(coeff, spectral_windows, degree_piecewise, wavelengths)

#         Y_pred[:, f_idx, :] = spectrum

#     return Y_pred

# Y_pred_val_piecewise = validate_piecewise_taylor(piecewise_gp, X_val_scaled_piecewise, y_scalers_piecewise)


# mre_piecewise = utils.mre_score(Y_val, Y_pred_val_piecewise, wavelengths, axis=None)
# mae_piecewise = utils.mae_score(Y_val, Y_pred_val_piecewise, wavelengths, axis=None)
# print("Piecewise Taylor GP MRE:", mre_piecewise)
# print("Piecewise Taylor GP MAE:", mae_piecewise)